In [1]:
import sys
import os
import torch
from torch.utils.data import DataLoader

In [2]:
sys.path.append("../src")

In [3]:
from models.vllm.blip2_model import Blip2Finetuner
from models.vllm.dataset_vllm import RAFCEInstructionDataset, collate_fn
from models.vllm.train_vllm import train_vision_llm

c:\Users\hedil\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [8]:
TRAIN_JSON = "../data/train_instructions.json"
VAL_JSON = "../data/val_instructions.json"
BATCH_SIZE = 2
NUM_EPOCHS = 3
LR = 1e-4

In [5]:
print("Initializing BLIP-2 with LoRA...")
model_wrapper = Blip2Finetuner(use_lora=True)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Initializing BLIP-2 with LoRA...
Loading BLIP-2 model: Salesforce/blip2-opt-2.7b...


`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/10.0G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Setting up LoRA configuration...
trainable params: 5,242,880 || all params: 3,750,004,736 || trainable%: 0.1398


In [9]:
print("Loading Datasets...")
train_ds = RAFCEInstructionDataset(TRAIN_JSON, model_wrapper.processor)
val_ds = RAFCEInstructionDataset(VAL_JSON, model_wrapper.processor)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

Loading Datasets...


In [ ]:
print("Starting Training...")
history = train_vision_llm(
    model_wrapper, 
    train_loader, 
    val_loader, 
    num_epochs=NUM_EPOCHS,
    learning_rate=LR,
    save_dir="../checkpoints/blip2_lora"
)
print("Training finished!")

Starting Training...
Epoch 1/3


Training:  21%|██        | 265/1270 [20:01:56<81:48:31, 293.05s/it, loss=0.104] 